# Legal Contract Clause Analyzer — Quick Demo
## Fine-tuned Qwen2.5-3B on CUAD (41 clause types)
**Run all 4 cells below to load and test the model (~3 min setup)**

In [ ]:
# CELL 1: Install & Setup (~2 min)
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# CELL 2: Load Model from HuggingFace Hub (~1 min)
from unsloth import FastLanguageModel
import torch, json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Vedant0824/legal-contract-clause-analyzer",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Model loaded! Ready to analyze contracts.")

In [ ]:
# CELL 3: Define the analyzer function
def analyze_clause(text):
    """Analyze a legal contract clause."""
    system = """You are a legal contract clause analyzer. Identify the clause type from 41 categories, extract key terms, assess risk (HIGH/MEDIUM/LOW), and explain in plain English. Respond in JSON."""
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Analyze this contract clause:\n\n{text}"},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    try:
        result = json.loads(response)
        print(json.dumps(result, indent=2))
    except:
        print(response)
    return response

print("Analyzer ready! Use: analyze_clause('your contract text here')")

In [ ]:
# CELL 4: Test with sample clauses (paste your own too!)

print("=" * 60)
print("TEST 1: Non-Compete Clause")
print("=" * 60)
analyze_clause("""The Employee agrees that during the term of employment and for a period 
of two (2) years following termination, Employee shall not directly or 
indirectly engage in any business that competes with the Company within 
a 50-mile radius of any Company office.""")

print("\n" + "=" * 60)
print("TEST 2: Governing Law")
print("=" * 60)
analyze_clause("""This Agreement shall be governed by and construed in accordance with 
the laws of the State of Delaware, without regard to its conflict of 
laws principles.""")

print("\n" + "=" * 60)
print("TEST 3: Liability Cap")
print("=" * 60)
analyze_clause("""IN NO EVENT SHALL EITHER PARTY'S TOTAL LIABILITY UNDER THIS AGREEMENT 
EXCEED THE TOTAL FEES PAID BY CUSTOMER DURING THE TWELVE (12) MONTH 
PERIOD IMMEDIATELY PRECEDING THE EVENT GIVING RISE TO SUCH LIABILITY.""")

In [ ]:
# CELL 5: TRY YOUR OWN! Paste any contract clause below:

your_clause = """PASTE ANY CONTRACT CLAUSE HERE"""

# Uncomment the line below and run:
# analyze_clause(your_clause)